# 01 — Exploración de Datos de Sensores Industriales

Este notebook explora el pipeline completo de datos del proyecto Smart-Industry:

- **Hot path** (tiempo real): InfluxDB `machine_stats` — datos de los últimos minutos
- **Cold path** (histórico): MinIO Parquet `datalake/clean/` — datos persistidos por Flink
- **Lambda query**: DuckDB unifica hot + cold en una sola consulta federada
- **ML Cloud**: IsolationForest vía FastAPI para detección de anomalías
- **Hash-Chain**: verificación de integridad criptográfica de mensajes

## Arquitectura del pipeline

```
sensor_simulator.py
      ↓ MQTT (mosquitto:1883)
mqtt_to_redpanda_bridge.py
      ↓ Redpanda topic: sensors_raw
Flink normalization job
      ↓ Redpanda topic: sensors_clean
Flink analytics job ──→ InfluxDB (bucket: sensores, measurement: machine_stats)
      ↓
Flink to MinIO job ──→ MinIO s3://datalake/clean/ (Parquet, particionado por fecha)
```

## Prerequisitos

El pipeline debe estar corriendo antes de ejecutar este notebook:

```bash
# Terminal 1 — Simulador
python src/01_ingestion/sensor_simulator.py --machines 5 --fault-rate 0.1

# Terminal 2 — Bridge MQTT → Redpanda
python src/01_ingestion/mqtt_to_redpanda_bridge.py

# Flink jobs (lanzados por init_pipeline.sh)
# - flink_normalization_job.py
# - flink_analytics_job.py
# - flink_to_minio_job.py

# Terminal 3 — FastAPI
uvicorn src.api.main:app --host 0.0.0.0 --port 8000 --reload
```

---

## Sección 1 — Conexión y configuración

In [ ]:
import os
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from influxdb_client import InfluxDBClient
from influxdb_client.client.warnings import MissingPivotFunction
import duckdb
import requests

# Suprimir advertencia de pivot en queries sin pivot() explícito
warnings.simplefilter('ignore', MissingPivotFunction)

# ── Configuración de conexiones ───────────────────────────────────────────────
# Dentro del devcontainer/workspace se usan hostnames internos de Docker
INFLUX_URL    = os.getenv('INFLUX_URL',    'http://influxdb:8086')
INFLUX_TOKEN  = os.getenv('INFLUX_TOKEN',  'supersecrettoken')
INFLUX_ORG    = os.getenv('INFLUX_ORG',    'ilerna')
INFLUX_BUCKET = os.getenv('INFLUX_BUCKET', 'sensores')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY',  'admin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY',  'Ilerna_Programaci0n')
MINIO_BUCKET     = 'datalake'
MINIO_PATH       = 'clean'

FASTAPI_URL = os.getenv('FASTAPI_URL', 'http://localhost:8000')

# ── Cliente InfluxDB ──────────────────────────────────────────────────────────
client    = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
query_api = client.query_api()

print(f'InfluxDB : {INFLUX_URL}')
print(f'MinIO    : {MINIO_ENDPOINT}')
print(f'FastAPI  : {FASTAPI_URL}')

---

## Sección 2 — InfluxDB: machine_stats (Hot Path)

El **hot path** contiene los datos de los últimos minutos almacenados en InfluxDB.

- **Bucket**: `sensores`
- **Measurement**: `machine_stats`
- **Campos**: `avg_temp_c`, `max_temp_c`, `count`, `alert`
- **Tag**: `device_id`

Flink analytics agrega las lecturas por ventanas de 1 minuto y escribe el resultado en InfluxDB.
El umbral de alerta es **80°C**: cuando `avg_temp_c > 80`, el campo `alert` vale `1`.

In [ ]:
# ── Consultar últimos 60 minutos de machine_stats ─────────────────────────────
RANGE = '60m'

query_hot = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -{RANGE})
  |> filter(fn: (r) => r._measurement == "machine_stats")
  |> filter(fn: (r) => r._field == "avg_temp_c")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "device_id", "avg_temp_c"])
'''

df_hot = query_api.query_data_frame(query_hot, org=INFLUX_ORG)

if not df_hot.empty:
    # Normalizar si query devuelve lista de DataFrames
    if isinstance(df_hot, list):
        df_hot = pd.concat(df_hot, ignore_index=True)
    df_hot = df_hot.rename(columns={'_time': 'ts'})
    df_hot['ts'] = pd.to_datetime(df_hot['ts'], utc=True)
    df_hot = df_hot.sort_values('ts')
    print(f'Registros hot path: {len(df_hot)}')
    print(f'Rango temporal: {df_hot["ts"].min()} → {df_hot["ts"].max()}')
    display(df_hot.head(10))
else:
    print('No hay datos en InfluxDB. Verifica que el pipeline está corriendo.')
    df_hot = pd.DataFrame(columns=['ts', 'device_id', 'avg_temp_c'])

In [ ]:
# ── Gráfica de líneas: avg_temp_c por device_id con umbral de alerta ──────────
if not df_hot.empty:
    fig = px.line(
        df_hot,
        x='ts',
        y='avg_temp_c',
        color='device_id',
        title='Temperatura media por máquina (últimos 60 min)',
        labels={'ts': 'Tiempo', 'avg_temp_c': 'Temp media (°C)', 'device_id': 'Máquina'},
        template='plotly_dark',
    )
    # Línea de umbral de alerta
    fig.add_hline(
        y=80,
        line_dash='dash',
        line_color='red',
        annotation_text='Umbral alerta 80°C',
        annotation_position='top left',
    )
    fig.update_layout(height=450)
    fig.show()
else:
    print('Sin datos para graficar.')

In [ ]:
# ── Gauges: temperatura actual por máquina ────────────────────────────────────
if not df_hot.empty:
    latest = df_hot.sort_values('ts').groupby('device_id').last().reset_index()

    fig = go.Figure()
    for _, row in latest.iterrows():
        fig.add_trace(go.Indicator(
            mode='gauge+number',
            value=round(row['avg_temp_c'], 1),
            title={'text': row['device_id']},
            gauge={
                'axis': {'range': [0, 120]},
                'bar': {'color': 'red' if row['avg_temp_c'] > 80 else 'green'},
                'threshold': {
                    'line': {'color': 'red', 'width': 3},
                    'thickness': 0.75,
                    'value': 80,
                },
            },
        ))

    n = len(latest)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig.update_layout(
        grid={'rows': rows, 'columns': cols, 'pattern': 'independent'},
        title='Temperatura actual por máquina',
        template='plotly_dark',
        height=250 * rows,
    )
    fig.show()
else:
    print('Sin datos para gauges.')

In [ ]:
# ── Tabla de estadísticas y conteo de alertas ─────────────────────────────────
# Consultar también el campo 'alert' para contar alarmas
query_stats = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -{RANGE})
  |> filter(fn: (r) => r._measurement == "machine_stats")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "device_id", "avg_temp_c", "max_temp_c", "count", "alert"])
'''

df_stats_raw = query_api.query_data_frame(query_stats, org=INFLUX_ORG)

if not df_stats_raw.empty:
    if isinstance(df_stats_raw, list):
        df_stats_raw = pd.concat(df_stats_raw, ignore_index=True)

    # Asegurar columnas presentes
    for col in ['avg_temp_c', 'max_temp_c', 'count', 'alert']:
        if col not in df_stats_raw.columns:
            df_stats_raw[col] = None

    stats_table = df_stats_raw.groupby('device_id').agg(
        temp_media=('avg_temp_c', 'mean'),
        temp_max=('max_temp_c', 'max'),
        lecturas=('count', 'sum'),
        alertas=('alert', 'sum'),
    ).round(2).reset_index()

    print('Resumen por máquina (últimos 60 min):')
    display(stats_table)
else:
    print('Sin datos para tabla de estadísticas.')

---

## Sección 3 — MinIO: Cold Path Parquet (Arquitectura Lambda)

El **cold path** almacena todos los datos históricos en MinIO como archivos Parquet.

Flink escribe continuamente en `s3://datalake/clean/` con **particionado Hive**:
```
datalake/clean/year=2026/month=03/day=12/hour=14/part-0001.parquet
```

**Arquitectura Lambda:**
- **Hot layer** (baja latencia): InfluxDB — consultas de los últimos minutos
- **Cold layer** (alta capacidad): MinIO Parquet — histórico completo
- **Serving layer**: DuckDB unifica ambas fuentes con `UNION ALL`

DuckDB puede leer Parquet directamente desde S3/MinIO usando la extensión `httpfs`.

In [ ]:
# ── Conectar DuckDB con MinIO (httpfs) y consultar Parquet ────────────────────
con = duckdb.connect()

# Cargar y configurar extensión httpfs para S3/MinIO
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    SET s3_endpoint='{MINIO_ENDPOINT}';
    SET s3_access_key_id='{MINIO_ACCESS_KEY}';
    SET s3_secret_access_key='{MINIO_SECRET_KEY}';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# Consultar datos del cold path
try:
    df_cold = con.execute("""
        SELECT *
        FROM read_parquet(
            's3://datalake/clean/**/*.parquet',
            hive_partitioning=true
        )
        LIMIT 1000
    """).df()

    print(f'Registros cold path (muestra): {len(df_cold)}')
    if not df_cold.empty:
        print(f'Columnas: {list(df_cold.columns)}')
        display(df_cold.head(10))
    else:
        print('El cold path aún no tiene datos. Flink necesita ~10 min para emitir el primer Parquet.')

except Exception as e:
    print(f'Cold path no disponible: {e}')
    print('Puede que Flink aún no haya escrito el primer archivo Parquet (rolling policy: ~10 min).')
    df_cold = pd.DataFrame()

In [ ]:
# ── Rango temporal y distribución de dispositivos en cold path ────────────────
if not df_cold.empty:
    # Detectar columna de tiempo
    time_col = next((c for c in ['ts', '_time', 'timestamp', 'event_time'] if c in df_cold.columns), None)
    device_col = next((c for c in ['device_id', 'machine_id'] if c in df_cold.columns), None)

    if time_col:
        df_cold[time_col] = pd.to_datetime(df_cold[time_col], utc=True)
        print(f'Rango temporal cold path: {df_cold[time_col].min()} → {df_cold[time_col].max()}')

    if device_col:
        dist = df_cold[device_col].value_counts().reset_index()
        dist.columns = ['device_id', 'registros']
        print('\nDistribución por dispositivo:')
        display(dist)

        fig = px.bar(
            dist,
            x='device_id',
            y='registros',
            title='Registros Parquet (cold path) por dispositivo',
            template='plotly_dark',
        )
        fig.show()
else:
    print('Cold path vacío: esperando primer archivo Parquet de Flink.')

---

## Sección 4 — Lambda Query: UNION ALL Hot + Cold

DuckDB permite hacer una **query federada** combinando datos del hot path (InfluxDB, cargado en memoria como DataFrame) y el cold path (MinIO Parquet, leído directamente con `httpfs`).

Esta es la clave de la **Arquitectura Lambda**:
- **Hot path** cubre los últimos minutos (baja latencia, datos recientes)
- **Cold path** cubre el histórico completo (alta capacidad, datos antiguos)
- `UNION ALL` en DuckDB fusiona ambas fuentes sin mover datos a un sistema centralizado

In [ ]:
# ── Query unificada: UNION ALL hot (InfluxDB) + cold (MinIO Parquet) ──────────
if not df_hot.empty:
    # Normalizar hot path para DuckDB
    hot_for_duck = df_hot[['ts', 'device_id', 'avg_temp_c']].copy()
    hot_for_duck = hot_for_duck.rename(columns={'avg_temp_c': 'temperature_c'})
    con.register('hot_data', hot_for_duck)

    # Construir query según disponibilidad del cold path
    if not df_cold.empty:
        # Detectar columnas del cold path
        time_col   = next((c for c in ['ts', '_time', 'timestamp'] if c in df_cold.columns), None)
        device_col = next((c for c in ['device_id', 'machine_id'] if c in df_cold.columns), None)
        temp_col   = next((c for c in ['avg_temp_c', 'temperature_c', 'temperature'] if c in df_cold.columns), None)

        if time_col and device_col and temp_col:
            cold_slim = df_cold[[time_col, device_col, temp_col]].copy()
            cold_slim.columns = ['ts', 'device_id', 'temperature_c']
            con.register('cold_data', cold_slim)

            df_unified = con.execute("""
                SELECT device_id, temperature_c, ts, 'InfluxDB (hot)' AS source
                FROM hot_data
                UNION ALL
                SELECT device_id, temperature_c, ts, 'MinIO (cold)' AS source
                FROM cold_data
                ORDER BY ts DESC
                LIMIT 5000
            """).df()
        else:
            df_unified = hot_for_duck.assign(source='InfluxDB (hot)').rename(columns={'temperature_c': 'temperature_c'})
    else:
        # Solo hot path disponible
        df_unified = con.execute("""
            SELECT device_id, temperature_c, ts, 'InfluxDB (hot)' AS source
            FROM hot_data
            ORDER BY ts DESC
        """).df()

    print(f'Registros unificados (Lambda Query): {len(df_unified)}')
    print(df_unified['source'].value_counts().to_string())

    if not df_unified.empty:
        df_unified['ts'] = pd.to_datetime(df_unified['ts'], utc=True)
        fig = px.line(
            df_unified.sort_values('ts'),
            x='ts',
            y='temperature_c',
            color='device_id',
            line_dash='source',
            title='Lambda Query — Hot + Cold (InfluxDB + MinIO Parquet)',
            labels={'ts': 'Tiempo', 'temperature_c': 'Temperatura (°C)', 'device_id': 'Máquina'},
            template='plotly_dark',
        )
        fig.add_hline(y=80, line_dash='dash', line_color='red', annotation_text='Umbral 80°C')
        fig.show()

        # Estadísticas combinadas
        combined_stats = df_unified.groupby(['device_id', 'source'])['temperature_c'].agg(
            media='mean', maximo='max', minimo='min', registros='count'
        ).round(2).reset_index()
        print('\nEstadísticas combinadas por máquina y fuente:')
        display(combined_stats)
else:
    print('Hot path vacío: no se puede ejecutar Lambda Query.')

---

## Sección 5 — IsolationForest: Entrenamiento y Predicción

El proyecto implementa dos niveles de inteligencia para detección de anomalías:

- **Edge intelligence** (Flink, tiempo real): regla determinista `avg_temp_c > 80°C → alert=1`. Baja latencia, sin estado ML.
- **Cloud intelligence** (FastAPI, bajo demanda): modelo **IsolationForest** entrenado con datos históricos de InfluxDB. Detecta patrones anómalos más sutiles y devuelve `failure_prob`.

El modelo vive en `src/api/anomaly_model.py` y se entrena vía `POST /model/train`.

### Endpoints relevantes

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| GET | `/model/status` | Estado del modelo (entrenado / no entrenado) |
| POST | `/model/train` | Entrena IsolationForest con datos de InfluxDB |
| GET | `/machines/{id}/predict` | Predice si una temperatura es anómala |

In [ ]:
# ── Verificar estado del modelo ───────────────────────────────────────────────
try:
    resp = requests.get(f'{FASTAPI_URL}/model/status', timeout=5)
    resp.raise_for_status()
    status = resp.json()
    print('Estado del modelo IsolationForest:')
    for k, v in status.items():
        print(f'  {k}: {v}')
except requests.exceptions.ConnectionError:
    print(f'FastAPI no accesible en {FASTAPI_URL}. Inicia con:')
    print('  uvicorn src.api.main:app --host 0.0.0.0 --port 8000 --reload')
except Exception as e:
    print(f'Error consultando model/status: {e}')

In [ ]:
# ── Entrenar el modelo con los últimos 60 minutos ─────────────────────────────
try:
    resp = requests.post(
        f'{FASTAPI_URL}/model/train',
        params={'range_minutes': 60, 'contamination': 0.1},
        timeout=30,
    )
    resp.raise_for_status()
    result = resp.json()
    print('Resultado del entrenamiento:')
    for k, v in result.items():
        print(f'  {k}: {v}')
except requests.exceptions.ConnectionError:
    print(f'FastAPI no accesible en {FASTAPI_URL}.')
except Exception as e:
    print(f'Error entrenando modelo: {e}')

In [ ]:
# ── Predicción para temperatura NORMAL (70°C) ─────────────────────────────────
MACHINE_ID = 'machine-001'
TEMP_NORMAL = 70.0

try:
    resp = requests.get(
        f'{FASTAPI_URL}/machines/{MACHINE_ID}/predict',
        params={'temperature_c': TEMP_NORMAL},
        timeout=5,
    )
    resp.raise_for_status()
    pred = resp.json()
    print(f'Prediccion para {MACHINE_ID} a {TEMP_NORMAL}°C (temperatura NORMAL):')
    for k, v in pred.items():
        print(f'  {k}: {v}')
except requests.exceptions.ConnectionError:
    print(f'FastAPI no accesible en {FASTAPI_URL}.')
except Exception as e:
    print(f'Error en prediccion: {e}')

In [ ]:
# ── Predicción para temperatura ANOMALA (95°C) ────────────────────────────────
TEMP_ANOMALA = 95.0

try:
    resp = requests.get(
        f'{FASTAPI_URL}/machines/{MACHINE_ID}/predict',
        params={'temperature_c': TEMP_ANOMALA},
        timeout=5,
    )
    resp.raise_for_status()
    pred = resp.json()
    print(f'Prediccion para {MACHINE_ID} a {TEMP_ANOMALA}°C (temperatura ANOMALA):')
    for k, v in pred.items():
        print(f'  {k}: {v}')
except requests.exceptions.ConnectionError:
    print(f'FastAPI no accesible en {FASTAPI_URL}.')
except Exception as e:
    print(f'Error en prediccion: {e}')

In [ ]:
# ── Predicciones masivas sobre los últimos 30 min de InfluxDB ─────────────────
# Consultar datos recientes para predecir anomalías en lote
query_batch = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -30m)
  |> filter(fn: (r) => r._measurement == "machine_stats")
  |> filter(fn: (r) => r._field == "avg_temp_c")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "device_id", "avg_temp_c"])
'''

df_batch = query_api.query_data_frame(query_batch, org=INFLUX_ORG)

if not df_batch.empty and isinstance(df_batch, list):
    df_batch = pd.concat(df_batch, ignore_index=True)

if not df_batch.empty:
    df_batch = df_batch.rename(columns={'_time': 'ts'})
    df_batch['ts'] = pd.to_datetime(df_batch['ts'], utc=True)

    # Llamar a FastAPI por cada fila
    failure_probs = []
    is_anomalies  = []

    for _, row in df_batch.iterrows():
        try:
            r = requests.get(
                f'{FASTAPI_URL}/machines/{row["device_id"]}/predict',
                params={'temperature_c': row['avg_temp_c']},
                timeout=3,
            )
            p = r.json()
            failure_probs.append(p.get('failure_prob', 0.0))
            is_anomalies.append(p.get('is_anomaly', False))
        except Exception:
            failure_probs.append(None)
            is_anomalies.append(None)

    df_batch['failure_prob'] = failure_probs
    df_batch['is_anomaly']   = is_anomalies

    print(f'Predicciones realizadas: {len(df_batch)}')
    print(f'Anomalias detectadas: {df_batch["is_anomaly"].sum()}')

    # Scatter plot coloreado por failure_prob
    df_plot = df_batch.dropna(subset=['failure_prob'])
    if not df_plot.empty:
        fig = px.scatter(
            df_plot,
            x='ts',
            y='avg_temp_c',
            color='failure_prob',
            facet_row='device_id',
            color_continuous_scale='RdYlGn_r',
            range_color=[0, 1],
            title='Predicciones IsolationForest — failure_prob por punto',
            labels={'ts': 'Tiempo', 'avg_temp_c': 'Temp media (°C)', 'failure_prob': 'P(fallo)'},
            template='plotly_dark',
            height=200 * df_plot['device_id'].nunique() + 100,
        )
        fig.add_hline(y=80, line_dash='dash', line_color='red')
        fig.show()
else:
    print('No hay datos para predicciones masivas (últimos 30 min vacíos).')

---

## Sección 6 — Hash Chain: Verificación de Integridad

El simulador implementa una **cadena de hashes SHA256** (inspirada en blockchain) para garantizar la integridad de los mensajes:

```
msg_1: { ..., prev_hash="0"×64,  hash=SHA256(content_1 + "0"×64) }
msg_2: { ..., prev_hash=hash_1,  hash=SHA256(content_2 + hash_1) }
msg_3: { ..., prev_hash=hash_2,  hash=SHA256(content_3 + hash_2) }
```

Si un atacante modifica `msg_2`, su hash cambia → `msg_3` detecta que `prev_hash` declarado ≠ hash real → el mensaje cae en la **Dead Letter Queue** `sensors_invalid` con `reason: hash_chain_broken`.

**Flink Hash Verifier** (`flink_hash_verifier_job.py`) mantiene un **keyed state** por `device_id` con el último hash visto, y valida cada mensaje nuevo antes de pasarlo a `sensors_verified`.

### Verificación desde línea de comandos

```bash
# Mensajes íntegros verificados
RP=$(docker ps -qf "label=com.docker.compose.service=redpanda")
docker exec $RP rpk topic consume sensors_verified -n 3

# Mensajes con cadena rota (DLQ)
docker exec $RP rpk topic consume sensors_invalid -n 3

# Publicar un mensaje con hash manipulado para forzar detección
mosquitto_pub -h mosquitto -p 1883 -t "sensors/telemetry" \\
  -m '{"device_id":"machine-001","temperature":72.0,"unit":"C",\
"ts":"2026-01-01T12:00:00Z","hash":"aaaaaa","prev_hash":"bbbbbb"}'
```

In [ ]:
# ── Verificar conteo de mensajes por tópico via FastAPI health ────────────────
# La verificación de hash chain es visible en los tópicos sensors_verified
# y sensors_invalid de Redpanda. Aquí consultamos el health de FastAPI
# y mostramos cómo interpretar los resultados.

try:
    resp = requests.get(f'{FASTAPI_URL}/health', timeout=5)
    resp.raise_for_status()
    health = resp.json()
    print('FastAPI health check:')
    for k, v in health.items():
        print(f'  {k}: {v}')
except requests.exceptions.ConnectionError:
    print(f'FastAPI no accesible en {FASTAPI_URL}.')
except Exception as e:
    print(f'Error en health check: {e}')

print()
print('Para verificar hash chain directamente, ejecuta en una terminal:')
print('  RP=$(docker ps -qf "label=com.docker.compose.service=redpanda")')
print('  docker exec $RP rpk topic consume sensors_verified -n 3')
print('  docker exec $RP rpk topic consume sensors_invalid -n 3')
print()
print('Campos esperados en sensors_verified:')
print('  { "device_id": "machine-001", "hash": "abc...", "prev_hash": "def...", ... }')
print()
print('Campos esperados en sensors_invalid (DLQ):')
print('  { "device_id": "machine-001", "reason": "hash_chain_broken", ... }')

In [ ]:
# ── Cierre de conexiones ──────────────────────────────────────────────────────
client.close()
con.close()
print('Conexiones cerradas. Notebook finalizado.')